# Upgrading gadgets for fault tolerance

<div style="border-left:4px solid #6d3f8c;background:#6d3f8c1a;border-radius:4px;margin:1em 0;"><div style="background:#6d3f8c;color:#ffffff;padding:0.35em 0.8em;font-weight:600;"><span style="display:inline-block;width:1.15em;height:1.15em;line-height:1.15em;border-radius:50%;background:#ffffff;color:#6d3f8c;text-align:center;font-weight:700;margin-right:0.5em;">!</span>Chapter focus</div>

<div style="padding:0.1em 1em;">

How do we redesign syndrome extraction and state preparation so every gadget reaches distance two?

</div>

</div>

## Learning objectives

After completing this unit, you will be able to:

- Upgrade the `syndrome` gadget to distance two without extra ancillas using an interleaved CNOT schedule.
- Verify with `effects_of` that spread ancilla faults now trigger parity checks.
- Explain how declared `flags` behave across `qodec`, `ec.filled`, `GadgetProfile.distance`, and `run_qir`.
- Upgrade state preparation gadgets to distance two with a `reject` flag and serialize a `.qodec.yaml` bundle.

## Setting up

The cell below starts from the baseline `bare-css/v1` protocol whose `syndrome` and preparation gadgets had distance one.

In [ ]:
import qodec
from qdk import ec
from _unit import make_baseline_protocol, show_stim_diagram

protocol = make_baseline_protocol()
gadgets = protocol.layers[0].gadgets

## Interleaving syndrome checks without extra ancillas

Fault tolerance does not prevent physical faults; it arranges each circuit so any single fault either acts harmlessly on the encoded state or triggers a parity check or flag. In *Analyzing Gadget Distance and Circuit Faults*, every distance-one witness in `syndrome` came from an ancilla fault spreading onto two data qubits whose two phase kicks canceled on the opposite ancilla. Following [Reichardt (arXiv:1804.06995) ↗](https://arxiv.org/abs/1804.06995), we can interleave the CNOTs of ancilla `4` ($X$-stabilizer) and ancilla `5` ($Z$-stabilizer) using the exact same two ancillas and eight CNOTs:

<div style="text-align:center;"><svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 960 302" width="100%" style="max-width:100%;height:auto" class="qdk-ec-interleaved" role="img" data-asset="tutorial_interleaved_syndrome.svg" aria-label="Comparison between bare syndrome extraction where X_4 spreads to X_2 X_3 with two canceling Z-ancilla flips (check unflipped), and Reichardt's interleaved CNOT schedule where CX 1 to 5 runs before X_4 so only X_3 flips ancilla q5 (check flipped, gadget distance 2)."> <style> .qdk-ec-interleaved { --qdk-fg: var(--vscode-editor-foreground, var(--jp-content-font-color1, currentColor)); --qdk-muted: color-mix(in srgb, var(--qdk-fg) 80%, transparent); --qdk-border: color-mix(in srgb, var(--qdk-fg) 30%, transparent); font-family: "Segoe UI", -apple-system, BlinkMacSystemFont, Roboto, Helvetica, Arial, sans-serif; } .qdk-ec-interleaved .cluster-blue { fill: color-mix(in srgb, #1976d2 8%, transparent); stroke: color-mix(in srgb, #1976d2 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-interleaved .cluster-teal { fill: color-mix(in srgb, #0d9488 8%, transparent); stroke: color-mix(in srgb, #0d9488 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-interleaved .cluster-purple { fill: color-mix(in srgb, #7c3aed 8%, transparent); stroke: color-mix(in srgb, #7c3aed 58%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-interleaved .cluster-neutral { fill: color-mix(in srgb, var(--qdk-fg) 4%, transparent); stroke: color-mix(in srgb, var(--qdk-fg) 32%, transparent); stroke-width: 1.8; } .qdk-ec-interleaved .cluster-red { fill: color-mix(in srgb, #dc2626 9%, transparent); stroke: color-mix(in srgb, #dc2626 62%, var(--qdk-fg)); stroke-width: 2; } .qdk-ec-interleaved .stage-blue { fill: color-mix(in srgb, #1976d2 18%, transparent); stroke: color-mix(in srgb, #1976d2 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-interleaved .stage-teal { fill: color-mix(in srgb, #0d9488 18%, transparent); stroke: color-mix(in srgb, #0d9488 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-interleaved .stage-purple { fill: color-mix(in srgb, #7c3aed 18%, transparent); stroke: color-mix(in srgb, #7c3aed 80%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-interleaved .stage-red { fill: color-mix(in srgb, #dc2626 16%, transparent); stroke: color-mix(in srgb, #dc2626 78%, var(--qdk-fg)); stroke-width: 1.8; } .qdk-ec-interleaved .loop-x { fill: color-mix(in srgb, #1976d2 20%, transparent); stroke: color-mix(in srgb, #1976d2 85%, var(--qdk-fg)); stroke-width: 2.2; } .qdk-ec-interleaved .loop-z { fill: color-mix(in srgb, #0d9488 20%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2.2; stroke-dasharray: 6 3; } .qdk-ec-interleaved .hdr-blue { fill: color-mix(in srgb, #1976d2 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-interleaved .hdr-teal { fill: color-mix(in srgb, #0d9488 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-interleaved .hdr-purple { fill: color-mix(in srgb, #7c3aed 70%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-interleaved .hdr-neutral { fill: var(--qdk-fg); font-size: 12.2px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-interleaved .hdr-red { fill: color-mix(in srgb, #dc2626 75%, var(--qdk-fg)); font-size: 12.5px; font-weight: 700; letter-spacing: 0.3px; } .qdk-ec-interleaved .box-title { fill: var(--qdk-fg); font-size: 11.6px; font-weight: 700; } .qdk-ec-interleaved .box-sub { fill: var(--qdk-muted); font-size: 10px; } .qdk-ec-interleaved .code-tag { fill: var(--qdk-fg); font-family: Consolas, "Courier New", monospace; font-size: 9.4px; font-weight: 600; } .qdk-ec-interleaved .badge-blue { fill: #1976d2; } .qdk-ec-interleaved .badge-teal { fill: #0d9488; } .qdk-ec-interleaved .badge-purple { fill: #7c3aed; } .qdk-ec-interleaved .badge-red { fill: #dc2626; } .qdk-ec-interleaved .badge-txt { fill: #ffffff; font-size: 9.6px; font-weight: 700; } .qdk-ec-interleaved .conn { stroke: color-mix(in srgb, var(--qdk-fg) 68%, transparent); stroke-width: 2.2; fill: none; } .qdk-ec-interleaved .wire { stroke: color-mix(in srgb, var(--qdk-fg) 48%, transparent); stroke-width: 1.5; fill: none; } .qdk-ec-interleaved .gate-neutral { stroke: color-mix(in srgb, var(--qdk-fg) 78%, transparent); stroke-width: 1.8; fill: none; } .qdk-ec-interleaved .gate-red { stroke: #dc2626; stroke-width: 2.3; fill: none; } .qdk-ec-interleaved .gate-teal { stroke: #0d9488; stroke-width: 2.4; fill: none; } .qdk-ec-interleaved .qubit-node { fill: var(--vscode-editor-background, #1e1e1e); stroke: color-mix(in srgb, var(--qdk-fg) 75%, transparent); stroke-width: 2; } .qdk-ec-interleaved .anc-node { fill: color-mix(in srgb, #0d9488 22%, transparent); stroke: color-mix(in srgb, #0d9488 85%, var(--qdk-fg)); stroke-width: 2; } </style> <rect x="12" y="12" width="458" height="278" rx="10" class="cluster-red"/> <text x="241" y="34" text-anchor="middle" class="hdr-red">BARE SCHEDULE &#8212; CHECK UNFLIPPED (DIST = 1)</text> <text x="26" y="66" class="code-tag">q0</text> <line x1="64" y1="62" x2="394" y2="62" class="wire"/> <text x="26" y="90" class="code-tag">q1</text> <line x1="64" y1="86" x2="394" y2="86" class="wire"/> <text x="26" y="114" class="code-tag">q2</text> <line x1="64" y1="110" x2="394" y2="110" class="wire"/> <text x="26" y="138" class="code-tag">q3</text> <line x1="64" y1="134" x2="394" y2="134" class="wire"/> <text x="22" y="168" class="code-tag">q4(X)</text> <line x1="64" y1="164" x2="394" y2="164" class="wire"/> <text x="22" y="198" class="code-tag">q5(Z)</text> <line x1="64" y1="194" x2="394" y2="194" class="wire"/> <line x1="86" y1="164" x2="86" y2="62" class="gate-neutral"/> <circle cx="86" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="86" cy="62" r="7" class="qubit-node"/> <text x="86" y="66" text-anchor="middle" class="code-tag">+</text> <line x1="114" y1="164" x2="114" y2="86" class="gate-neutral"/> <circle cx="114" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="114" cy="86" r="7" class="qubit-node"/> <text x="114" y="90" text-anchor="middle" class="code-tag">+</text> <rect x="132" y="154" width="40" height="20" rx="4" class="badge-red"/> <text x="152" y="168" text-anchor="middle" class="badge-txt">X_4</text> <line x1="190" y1="164" x2="190" y2="110" class="gate-red"/> <circle cx="190" cy="164" r="4.5" fill="#dc2626"/> <circle cx="190" cy="110" r="7" class="qubit-node"/> <text x="190" y="114" text-anchor="middle" class="code-tag">+</text> <line x1="220" y1="164" x2="220" y2="134" class="gate-red"/> <circle cx="220" cy="164" r="4.5" fill="#dc2626"/> <circle cx="220" cy="134" r="7" class="qubit-node"/> <text x="220" y="138" text-anchor="middle" class="code-tag">+</text> <rect x="242" y="184" width="18" height="20" rx="3" class="qubit-node"/> <text x="251" y="198" text-anchor="middle" class="code-tag">H</text> <line x1="274" y1="62" x2="274" y2="194" class="gate-neutral"/> <circle cx="274" cy="62" r="4" fill="var(--qdk-fg)"/> <circle cx="274" cy="194" r="4" fill="var(--qdk-fg)"/> <line x1="300" y1="86" x2="300" y2="194" class="gate-neutral"/> <circle cx="300" cy="86" r="4" fill="var(--qdk-fg)"/> <circle cx="300" cy="194" r="4" fill="var(--qdk-fg)"/> <line x1="328" y1="110" x2="328" y2="194" class="gate-red" stroke-dasharray="4 2"/> <circle cx="328" cy="110" r="4.5" fill="#dc2626"/> <circle cx="328" cy="194" r="4.5" fill="#dc2626"/> <line x1="356" y1="134" x2="356" y2="194" class="gate-red" stroke-dasharray="4 2"/> <circle cx="356" cy="134" r="4.5" fill="#dc2626"/> <circle cx="356" cy="194" r="4.5" fill="#dc2626"/> <rect x="370" y="184" width="18" height="20" rx="3" class="qubit-node"/> <text x="379" y="198" text-anchor="middle" class="code-tag">H</text> <rect x="398" y="101" width="60" height="18" rx="4" class="badge-red"/> <text x="428" y="114" text-anchor="middle" class="badge-txt">X_2</text> <rect x="398" y="125" width="60" height="18" rx="4" class="badge-red"/> <text x="428" y="138" text-anchor="middle" class="badge-txt">X_3</text> <rect x="398" y="185" width="60" height="18" rx="4" class="stage-red"/> <text x="428" y="198" text-anchor="middle" class="code-tag">&#916;chk = 0</text> <rect x="26" y="218" width="430" height="58" rx="7" class="stage-red"/> <text x="241" y="239" text-anchor="middle" class="box-title">Both X_2 and X_3 flip q5 &#8594; two flips cancel (&#916;M5 = 0)</text> <text x="241" y="260" text-anchor="middle" class="code-tag">Leaves logical error X_2 X_3 (out[0].z[0]) &#183; checks = []</text> <rect x="490" y="12" width="458" height="278" rx="10" class="cluster-teal"/> <text x="719" y="34" text-anchor="middle" class="hdr-teal">INTERLEAVED SCHEDULE &#8212; CHECK FLIPPED (DIST = 2)</text> <text x="504" y="66" class="code-tag">q0</text> <line x1="542" y1="62" x2="872" y2="62" class="wire"/> <text x="504" y="90" class="code-tag">q1</text> <line x1="542" y1="86" x2="872" y2="86" class="wire"/> <text x="504" y="114" class="code-tag">q2</text> <line x1="542" y1="110" x2="872" y2="110" class="wire"/> <text x="504" y="138" class="code-tag">q3</text> <line x1="542" y1="134" x2="872" y2="134" class="wire"/> <text x="500" y="168" class="code-tag">q4(X)</text> <line x1="542" y1="164" x2="872" y2="164" class="wire"/> <text x="500" y="198" class="code-tag">q5(Z)</text> <line x1="542" y1="194" x2="872" y2="194" class="wire"/> <line x1="562" y1="164" x2="562" y2="62" class="gate-neutral"/> <circle cx="562" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="562" cy="62" r="7" class="qubit-node"/> <text x="562" y="66" text-anchor="middle" class="code-tag">+</text> <line x1="590" y1="110" x2="590" y2="194" class="gate-neutral"/> <circle cx="590" cy="110" r="4" fill="var(--qdk-fg)"/> <circle cx="590" cy="194" r="7" class="qubit-node"/> <text x="590" y="198" text-anchor="middle" class="code-tag">+</text> <line x1="618" y1="62" x2="618" y2="194" class="gate-neutral"/> <circle cx="618" cy="62" r="4" fill="var(--qdk-fg)"/> <circle cx="618" cy="194" r="7" class="qubit-node"/> <text x="618" y="198" text-anchor="middle" class="code-tag">+</text> <line x1="646" y1="86" x2="646" y2="194" class="gate-neutral"/> <circle cx="646" cy="86" r="4" fill="var(--qdk-fg)"/> <circle cx="646" cy="194" r="7" class="qubit-node"/> <text x="646" y="198" text-anchor="middle" class="code-tag">+</text> <line x1="674" y1="164" x2="674" y2="110" class="gate-neutral"/> <circle cx="674" cy="164" r="4" fill="var(--qdk-fg)"/> <circle cx="674" cy="110" r="7" class="qubit-node"/> <text x="674" y="114" text-anchor="middle" class="code-tag">+</text> <rect x="692" y="154" width="40" height="20" rx="4" class="badge-red"/> <text x="712" y="168" text-anchor="middle" class="badge-txt">X_4</text> <line x1="752" y1="164" x2="752" y2="134" class="gate-red"/> <circle cx="752" cy="164" r="4.5" fill="#dc2626"/> <circle cx="752" cy="134" r="7" class="qubit-node"/> <text x="752" y="138" text-anchor="middle" class="code-tag">+</text> <line x1="784" y1="164" x2="784" y2="86" class="gate-red"/> <circle cx="784" cy="164" r="4.5" fill="#dc2626"/> <circle cx="784" cy="86" r="7" class="qubit-node"/> <text x="784" y="90" text-anchor="middle" class="code-tag">+</text> <line x1="824" y1="134" x2="824" y2="194" class="gate-teal"/> <circle cx="824" cy="134" r="4.8" fill="#0d9488"/> <circle cx="824" cy="194" r="7" class="qubit-node"/> <text x="824" y="198" text-anchor="middle" class="code-tag">+</text> <rect x="876" y="77" width="60" height="18" rx="4" class="badge-red"/> <text x="906" y="90" text-anchor="middle" class="badge-txt">X_1</text> <rect x="876" y="125" width="60" height="18" rx="4" class="badge-red"/> <text x="906" y="138" text-anchor="middle" class="badge-txt">X_3</text> <rect x="876" y="185" width="60" height="18" rx="4" class="badge-teal"/> <text x="906" y="198" text-anchor="middle" class="badge-txt">&#916;chk = 1</text> <rect x="504" y="218" width="430" height="58" rx="7" class="stage-teal"/> <text x="719" y="239" text-anchor="middle" class="box-title">CX 1&#8594;5 ran early, so only X_3 hits CX 3&#8594;5 to flip q5</text> <text x="719" y="260" text-anchor="middle" class="code-tag">Flips checks[1], checks[3] &#183; detected (gadget dist = 2)</text> </svg></div>

<div style="text-align:center;">

*Left: in the sequential schedule (`H/R` gates omitted for clarity), midpoint $X_4$ spreads to $X_2 X_3$ before ancilla `5` couples, canceling on `q5` (`Δchk = 0`). Right: in Reichardt's interleaved schedule, `CX 2 5` has already run before a spreading $X_4$ fault, so only one of the two data flips hits a remaining `q5` CNOT (`Δchk = 1`, detected).*

</div>

After replacing `gadgets["syndrome"].circuit.source`, we call `ec.filled` to recompute the parity checks, verify that the gadget distance rises to `2`, and check `ec.audit(protocol)`:

In [ ]:
reichardt_source = """\
R 4 5
H 4
CX 4 0
CX 2 5
CX 0 5
CX 1 5
CX 4 2
CX 4 3
CX 4 1
CX 3 5
H 4
M 4 5
"""
gadgets["syndrome"].circuit.source = reichardt_source
gadgets["syndrome"] = ec.filled(gadgets["syndrome"])

improved_syn = ec.GadgetProfile(gadgets["syndrome"])
print("Upgraded syndrome distance:", improved_syn.distance())
print(ec.audit(protocol))
show_stim_diagram(reichardt_source)

### Trace faults through the interleaved circuit

Why does the interleaved schedule reach distance `2`? An ancilla fault can still spread onto two data qubits, *but the interleaved gates cause that fault to flip the other ancilla's readout as well*. We can confirm this directly with `effects_of`:

In [ ]:
calls = list(gadgets["syndrome"].circuit.calls())
test_faults = [
    ec.FaultEvent.after(5, ec.Pauli("Z_5")),
    ec.FaultEvent.after(7, ec.Pauli("X_4")),
]
for fault, effect in zip(test_faults, improved_syn.effects_of(test_faults)):
    call = calls[fault.locations[0].after_call]
    print(f"{fault} (after {call.mnemonic} {list(call.operands)}) -> {effect}")

## Adding a rejection flag to state preparation

The baseline `prepare_z_all` and `prepare_x_all` gadgets initialize $\vert 00\rangle_L$ and $\vert ++\rangle_L$ by non-fault-tolerantly measuring stabilizers, leaving them at distance `1`. To reach distance `2`, we use a **flag ancilla**—so it is worth understanding precisely what a flag means across the QDK error correction layers:
- **Protocol contract (`qodec`):** An instruction declares named flags on `instruction.flags` (for example, `["reject"]`), and its gadget binds each flag at the **end of `gadget.readouts`**—after any logical observable readouts—as a single-key mapping such as `{"reject": ["circuit.readouts[0]"]}`.
- **Noiseless semantics and `ec.filled` (`qdk.ec`):** On a fault-free execution with a trivial incoming frame, every flag parity must evaluate to `0` (`ec.audit` verifies this invariant). Because flags exist to herald noisy circuit faults rather than noiseless logical observables, `ec.filled` preserves whatever flags you declare instead of inferring them.
- **Gadget distance (`GadgetProfile.distance`):** Any fault combination that flips at least one declared `check` **or** `flag` is treated as detected and excluded from counting as an undetected logical error (without assuming a specific decoder; two faults can still cancel a flag at $O(p^2)$).
- **Runtime execution (`qdk.simulation.run_qir`):** When the calling program captures a flag as a return value (for example, a Q# `@Measurement()` returning `Result`), the flag bit is passed directly to caller logic. When a raised flag (`1`) is **not** returned to the caller, the runtime triggers `on_shot_failure` (`"discard"` by default).

For `prepare_z_all`, we prepare the four-qubit GHZ state unitarily from control qubit `0` (`H 0` followed by `CX 0 1`, `CX 0 2`, `CX 0 3`) and bracket that fan-out with two CNOTs onto flag ancilla `4` (`CX 0 4` before and after). An $X_0$ fault *before* the first `CX 0 4` spreads to $X_0 X_1 X_2 X_3$ (a harmless stabilizer on $\vert 00\rangle_L$, leaving `M 4 = 0`), whereas any $X_0$ fault *between* the two `CX 0 4` gates that spreads to a weight-two logical error flips `M 4 = 1` and raises `reject`:

In [ ]:
flagged_prep_z = """\
R 0 1 2 3 4
H 0
CX 0 4
CX 0 1
CX 0 2
CX 0 3
CX 0 4
M 4
"""
prep_z = gadgets["prepare_z_all"]
prep_z.implements.flags = ["reject"]
prep_z.circuit.source = flagged_prep_z
prep_z.readouts = [{"reject": ["circuit.readouts[0]"]}]
prep_z.checks = []
gadgets["prepare_z_all"] = ec.filled(prep_z)

print("prepare_z_all distance:", ec.GadgetProfile(gadgets["prepare_z_all"]).distance())
print(ec.audit(protocol))
show_stim_diagram(flagged_prep_z)

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;How does Reichardt's interleaved syndrome circuit reach gadget distance two without adding a flag qubit?</summary>

<div style="padding:0.1em 1em;">

By interleaving the CNOTs of ancillas `4` and `5`, any midpoint fault on one ancilla that spreads to two data qubits overlaps an odd number of interleaved gates on the opposite ancilla, flipping its stabilizer check.

</div>

</details></div>

<div style="border-left:4px solid #8c4a00;background:#8c4a001a;border-radius:4px;margin:1em 0;"><details><summary style="background:#8c4a00;color:#ffffff;padding:0.35em 0.8em;cursor:pointer;font-weight:600;">&#10067;&nbsp;Why must you bind the reject flag readout manually before calling ec.filled on state preparation?</summary>

<div style="padding:0.1em 1em;">

`ec.filled` derives checks and observable readouts from noiseless simulation. In a noiseless run, ancilla `4` is deterministically `0`, so only the protocol author can specify that `circuit.readouts[0]` serves as the `reject` flag rather than an internal parity check.

</div>

</details></div>

## Exercise: Upgrade X-basis preparation with a flag ancilla

Preparing $\vert ++\rangle_L$ (`prepare_x_all`) uses the exact same flagged GHZ circuit as `prepare_z_all`, followed by transversal `H 0 1 2 3` before measuring `M 4`. Complete `upgrade_prepare_x_distance` by filling the gadget with `ec.filled` and returning its integer gadget distance.

In [ ]:
from qdk import ec
from _unit import exercise, make_upgraded_syndrome_protocol


@exercise
def upgrade_prepare_x_distance() -> int:
    proto = make_upgraded_syndrome_protocol()
    prep_x = proto.layers[0].gadgets['prepare_x_all']
    prep_x.implements.flags = ['reject']
    prep_x.circuit.source = (
        'R 0 1 2 3 4\nH 0\nCX 0 4\nCX 0 1\nCX 0 2\n'
        'CX 0 3\nCX 0 4\nH 0 1 2 3\nM 4\n'
    )
    prep_x.readouts = [{'reject': ['circuit.readouts[0]']}]
    prep_x.checks = []
    # Derive the checks with ec.filled and return the upgraded gadget distance (.value)
    return ...

**Hint**

With `flags` and `readouts` bound, `ec.filled` derives the output stabilizer checks. Call `upgraded = ec.filled(prep_x)` and return `ec.GadgetProfile(upgraded).distance().value`.

In [ ]:
from qdk import ec
from _unit import exercise, make_upgraded_syndrome_protocol


@exercise
def upgrade_prepare_x_distance() -> int:
    proto = make_upgraded_syndrome_protocol()
    prep_x = proto.layers[0].gadgets['prepare_x_all']
    prep_x.implements.flags = ['reject']
    prep_x.circuit.source = (
        'R 0 1 2 3 4\nH 0\nCX 0 4\nCX 0 1\nCX 0 2\n'
        'CX 0 3\nCX 0 4\nH 0 1 2 3\nM 4\n'
    )
    prep_x.readouts = [{'reject': ['circuit.readouts[0]']}]
    prep_x.checks = []
    upgraded = ec.filled(prep_x)
    return ec.GadgetProfile(upgraded).distance().value

`2`. Bracketing the fan-out from qubit `0` with `CX 0 4` gates heralds any spreading fault on `M 4`, and `H 0 1 2 3` maps the stabilized $+Z_0, +Z_1$ state to $+X_0, +X_1$.

## Saving and reloading the upgraded protocol

Once all six gadgets reach distance two and pass `ec.audit`, use `protocol.save(..., single_file=True)` to serialize the manifest, codes, instruction sets, and gadgets into one portable YAML bundle:

In [ ]:
from tempfile import TemporaryDirectory
from _unit import make_upgraded_protocol

upgraded_protocol = make_upgraded_protocol()
print("Audit:", ec.audit(upgraded_protocol))
for name, g in sorted(upgraded_protocol.layers[0].gadgets.items()):
    print(f"  {name:<16} -> distance {ec.GadgetProfile(g).distance()}")

with TemporaryDirectory() as tmpdir:
    bundle_path = upgraded_protocol.save(tmpdir, single_file=True)
    assert qodec.Qodec.load(bundle_path) == upgraded_protocol
    print("Saved and verified round-trip bundle:", bundle_path.name)

## Further reading

- [Fault-tolerant quantum error correction for Steane's seven-qubit color code with few or no extra qubits (Reichardt, arXiv:1804.06995) ↗](https://arxiv.org/abs/1804.06995)

<div style="display:flex;justify-content:space-between;align-items:stretch;gap:12px;margin-top:8px"><a href="../05-circuit-distance/circuit_distance.workbook.ipynb" title="Previous unit: Analyzing Gadget Distance and Circuit Faults" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-secondaryBackground, transparent);color:var(--vscode-button-secondaryForeground, inherit);border-color:var(--vscode-widget-border, #8884)"><span aria-hidden="true">&#8249;</span><span><span style="display:block;font-size:11px;opacity:.75">Previous unit</span><span style="display:block;font-weight:600">Analyzing Gadget Distance and Circuit Faults</span></span></a><a href="../07-qsharp-compilation/qsharp_compilation.workbook.ipynb" title="Next unit: Compiling Logical Experiments from Q#" style="display:inline-flex;align-items:center;gap:8px;padding:6px 12px;border:1px solid;border-radius:4px;text-decoration:none;background:var(--vscode-button-background, #005fb8);color:var(--vscode-button-foreground, #ffffff);border-color:var(--vscode-button-background, #005fb8)"><span style="text-align:right"><span style="display:block;font-size:11px;opacity:.75">Next unit</span><span style="display:block;font-weight:600">Compiling Logical Experiments from Q#</span></span><span aria-hidden="true">&#8250;</span></a></div>